# Spark, on the demo's own sandbox

This notebook runs on **the demo sandbox**, not the platform's shared Spark server:
one pod, `local[2]` (two cores, 2 GiB), restarted every night at 03:00 UTC. It is
shared by everyone who signs in with `demo`, so keep the queries small — and
nothing you do here is private.

The catalogs are **read-only** for visitors. The data is the shop's `gold` layer,
loaded daily by the demo's pipelines. There is no internet from this sandbox.

In [ ]:
print(spark)  # the session every kernel has — opened on first use
spark.range(10).count()

In [ ]:
daily = spark.table("gold.shop.daily_revenue").orderBy("order_date", ascending=False)
display(daily.limit(10))

One row per day: the number of order lines, the units sold and the revenue.
The nightly `shop` pipeline fills this table from `silver.shop.stg_order_lines`.

In [ ]:
from pyspark.sql import functions as F

mix = spark.table("gold.shop.product_mix")
per_category = mix.groupBy("category").agg(
    F.sum("revenue").alias("category_revenue"),
    F.sum("units").alias("category_units"),
)
joined = (mix.join(per_category, on="category")
            .withColumn("share_of_category",
                        F.round(F.col("revenue") / F.col("category_revenue"), 3))
            .orderBy(F.desc("revenue")))
display(joined.select("product_name", "category", "units", "revenue",
                      "share_of_category", "rank_in_category").limit(15))

In [ ]:
%%sql
SELECT category, round(sum(revenue), 2) AS revenue, sum(units) AS units,
       count(*) AS products
FROM gold.shop.product_mix
GROUP BY category
ORDER BY revenue DESC

Two engines, one catalog: the DataFrame above ran on the sandbox's Spark, the SQL
cell through Trino. Both see the same Iceberg tables.

What to try next: `spark.table("gold.shop.customer_ltv")`, or the SQL quickstart
(`warehouse-quickstart.ipynb`) for Trino-only work.

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN gold.shop").show(truncate=False)